# 16: Why Sequences Matter - The Order Problem

## The Problem with Everything So Far

All our previous models treat inputs as **independent**:
- Bag of words: "cat dog" = "dog cat"
- Feed-forward networks: no memory of previous inputs
- Embeddings: capture word meaning, not sentence structure

But language is **sequential**! Order matters:
- "Dog bites man" ≠ "Man bites dog"
- "not bad" ≠ "bad"
- "I didn't like the movie" (negative despite "like")

### The Web Dev Analogy

Sequences are like **event streams**:
- Previous events affect current state
- Order matters (click → hover ≠ hover → click)
- Need to maintain context over time

## What You'll Learn
- [ ] Explain why word order matters for meaning
- [ ] Identify types of sequence tasks (many-to-one, many-to-many, etc.)
- [ ] Demonstrate specific failures of bag-of-words on order-dependent examples

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 2/5**: Bag-of-words treats text as an unordered set | "dog bites man" = "man bites dog" in BoW — clearly wrong! |
| **Lesson 13**: Embeddings capture word meaning | But embeddings alone don't capture *order* — we need sequence models |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to explore sequences! 🔄")

## 1. Sequences Are Everywhere

Sequential data appears in many domains:

In [ ]:
print("Sequential Data Examples:")
print("=" * 50)
print("\n📝 Text:")
print("   'The cat sat on the mat'")
print("   Each word depends on previous words")

print("\n🎵 Music:")
print("   Notes played in sequence create melody")
print("   Same notes, different order = different song")

print("\n📈 Time Series:")
print("   Stock prices, weather, sensor data")
print("   Current value influenced by past values")

print("\n🎬 Video:")
print("   Frames in sequence tell a story")
print("   Shuffling frames = nonsense")

print("\n💬 Conversation:")
print("   Each utterance depends on context")
print("   'Yes' means different things in different contexts")

## 2. The Problem with Bag-of-Words

In [ ]:
# Simple vocabulary
vocab = {'not': 0, 'good': 1, 'bad': 2, 'movie': 3}

def bag_of_words(sentence, vocab):
    """Convert sentence to bag-of-words vector."""
    vec = np.zeros(len(vocab))
    for word in sentence.lower().split():
        if word in vocab:
            vec[vocab[word]] += 1
    return vec

# Two sentences with OPPOSITE meanings
s1 = "good movie"
s2 = "not good movie"

bow1 = bag_of_words(s1, vocab)
bow2 = bag_of_words(s2, vocab)

print(f"Sentence 1: '{s1}'")
print(f"BoW: {bow1}")
print(f"Sentiment: POSITIVE ✓")

print(f"\nSentence 2: '{s2}'")
print(f"BoW: {bow2}")
print(f"Sentiment: NEGATIVE (but BoW doesn't capture 'not'!)")

print("\n❌ Bag-of-words loses word order!")
print("   'not good' and 'good' both have 'good' → confusing!")

## 3. Order Changes Meaning

In [ ]:
# Examples where order completely changes meaning
examples = [
    ("Dog bites man", "Man bites dog", "Different events!"),
    ("The movie was not bad", "The movie was bad", "Opposite sentiments!"),
    ("I can't complain", "I can complain", "Negation matters!"),
    ("Stop right there", "Right there stop", "One natural, one awkward"),
]

print("Order Changes Everything:")
print("=" * 70)
for s1, s2, note in examples:
    print(f"\n✓ '{s1}'")
    print(f"✗ '{s2}'")
    print(f"  → {note}")
    
    # Show they have same bag-of-words
    words1 = set(s1.lower().split())
    words2 = set(s2.lower().split())
    if words1 == words2:
        print(f"  → Same words! BoW can't distinguish.")

## 4. Context Windows: A First Attempt

In [ ]:
# N-grams: look at consecutive word pairs
def extract_bigrams(sentence):
    """Extract pairs of consecutive words."""
    words = sentence.lower().split()
    bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
    return bigrams

s1 = "the movie was not bad"
s2 = "the movie was bad"

print(f"Sentence 1: '{s1}'")
print(f"Bigrams: {extract_bigrams(s1)}")
print("   → Captures 'not bad' together! ✓")

print(f"\nSentence 2: '{s2}'")
print(f"Bigrams: {extract_bigrams(s2)}")
print("   → Different bigrams! ✓")

print("\n💡 Bigrams are better than bag-of-words!")
print("   But...")
print("   - Exponential vocabulary growth")
print("   - Limited to fixed window (2 words)")
print("   - Doesn't scale to longer dependencies")

## 5. The Challenge: Long-Range Dependencies

In [ ]:
# Examples requiring long-range context
long_range = [
    (
        "The player, who had been injured for months, finally scored.",
        "'scored' relates to 'player' (10 words away!)"
    ),
    (
        "Although I initially disliked it, the movie grew on me.",
        "'grew on me' reverses 'disliked' (7 words away)"
    ),
    (
        "The book, despite its length and complexity, was amazing.",
        "'was' refers back to 'book' (7 words away)"
    ),
]

print("Long-Range Dependencies:")
print("=" * 70)
for sentence, explanation in long_range:
    print(f"\n'{sentence}'")
    print(f"  → {explanation}")
    
print("\n❌ N-grams can't handle this!")
print("   Would need huge N for long sentences")
print("   → Sparse, impractical")

print("\n✅ Need: A model with MEMORY")

## 6. What We Need: Sequential Models

Requirements for processing sequences:
1. **Process inputs in order** (word by word)
2. **Maintain memory** of previous inputs
3. **Variable length** sequences
4. **Capture dependencies** across time

This is exactly what **Recurrent Neural Networks (RNNs)** do!

In [ ]:
# Visualize sequential processing
sentence = "not bad movie"
words = sentence.split()

print("Sequential Processing:")
print("=" * 50)
print("\nStep-by-step:")

context = []
for i, word in enumerate(words):
    context.append(word)
    print(f"\nStep {i+1}: Read '{word}'")
    print(f"  Context so far: {' '.join(context)}")
    print(f"  Model now knows: {' '.join(context)}")

print("\n✅ By the end, model has seen entire sequence IN ORDER")
print("   Can use full context for prediction!")

## 7. Sequence Tasks

In [ ]:
print("Common Sequence Tasks:")
print("=" * 70)

print("\n1. Sequence Classification (many-to-one)")
print("   Input: sequence of words")
print("   Output: single label")
print("   Example: Sentiment analysis")
print("   'This movie is great' → POSITIVE")

print("\n2. Sequence Labeling (many-to-many, aligned)")
print("   Input: sequence of words")
print("   Output: label for each word")
print("   Example: Part-of-speech tagging")
print("   'The cat sat' → [DET, NOUN, VERB]")

print("\n3. Sequence Generation (one-to-many)")
print("   Input: single input (or none)")
print("   Output: sequence")
print("   Example: Image captioning, text generation")
print("   [image] → 'A cat sitting on a couch'")

print("\n4. Sequence-to-Sequence (many-to-many, not aligned)")
print("   Input: sequence")
print("   Output: different length sequence")
print("   Example: Translation")
print("   'Hello world' → 'Bonjour le monde'")

## 8. Preview: The RNN Solution

In [ ]:
# Conceptual visualization of RNN processing
fig, ax = plt.subplots(figsize=(12, 6))

words = ['The', 'movie', 'was', 'not', 'bad']
n_words = len(words)

# Draw RNN cells
for i in range(n_words):
    # Cell
    rect = plt.Rectangle((i*2, 1), 1.5, 1, 
                         fill=True, facecolor='lightblue', 
                         edgecolor='blue', linewidth=2)
    ax.add_patch(rect)
    
    # Word input
    ax.text(i*2 + 0.75, 0.5, words[i], 
           ha='center', va='center', fontsize=12, fontweight='bold')
    ax.arrow(i*2 + 0.75, 0.7, 0, 0.2, 
            head_width=0.1, head_length=0.05, fc='black', ec='black')
    
    # RNN label
    ax.text(i*2 + 0.75, 1.5, 'RNN', 
           ha='center', va='center', fontsize=10, color='darkblue')
    
    # Hidden state arrow (except last)
    if i < n_words - 1:
        ax.arrow(i*2 + 1.6, 1.5, 0.3, 0, 
                head_width=0.15, head_length=0.1, 
                fc='red', ec='red', linewidth=2)
        ax.text(i*2 + 1.75, 1.9, 'memory', 
               ha='center', fontsize=8, color='red')

# Output
ax.arrow((n_words-1)*2 + 0.75, 2.1, 0, 0.3, 
        head_width=0.15, head_length=0.1, fc='green', ec='green', linewidth=2)
ax.text((n_words-1)*2 + 0.75, 2.6, 'Output: NEGATIVE', 
       ha='center', fontsize=12, fontweight='bold', color='green')

ax.set_xlim(-0.5, n_words*2)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('RNN Processing Sequence (with memory flow)', 
            fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n💡 Key insight: Each RNN cell passes 'memory' to the next!")
print("   By 'bad', the model remembers seeing 'not' before it.")

## 📝 Check Your Understanding

1. Why does order matter in language?
2. What's wrong with bag-of-words for "not good"?
3. What are long-range dependencies?
4. What makes n-grams impractical for long sequences?
5. What is the key capability RNNs need to have?

In [ ]:
# --- Quick Check: Why Order Matters ---
# Does bag-of-words distinguish "dog bites man" from "man bites dog"?
# a) Yes, BoW captures word order
# b) No, BoW gives identical vectors for both
# c) Only if we use bigrams
# d) Only with TF-IDF weighting

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "BoW counts words regardless of order — both have the same word counts!"
print("Exercise 1 passed! ✓")

# --- Quick Check: Sequence Task Types ---
# Sentiment analysis (text → positive/negative) is what type of sequence task?
# a) one-to-one (single input → single output)
# b) many-to-one (sequence → single label)
# c) one-to-many (single input → sequence)
# d) many-to-many (sequence → sequence)

your_answer_2 = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer_2 is not None, "Pick an answer!"
assert your_answer_2 == 'b', "A full review (many words) maps to one label (positive/negative) = many-to-one!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

**Sequences are fundamental to language**:
- Order changes meaning completely
- Bag-of-words and n-grams are insufficient
- Need models with **memory** of previous inputs

**Key problems to solve**:
1. Variable-length inputs
2. Long-range dependencies
3. Order-dependent processing

**Solution**: Recurrent Neural Networks (RNNs)
- Process sequences step-by-step
- Maintain hidden state (memory)
- Pass context forward

**Next up**: Building our first RNN! →